# NB 35 — Configurable memory / prompt-assembly demo (offline)

**No LLM calls, no network, no data files.** This notebook makes the v0.8
configurable-memory feature (`docs/Model_Design.md` §30, `USER_GUIDE.md` v0.8
config path) concrete for a researcher who wants to run *memory-ablation*
experiments — e.g. "does removing the Day-0 anchor stop agents locking in
after Day 1?".

It builds one detached `SurveyedCitizen` with a hand-populated 3-day memory
(the same style as `tests/test_memory.py`), then shows how the **exact
assembled system-prompt context** changes when you swap the `memory` config
preset — the same knob you pass via `SIM_CONFIG["memory"]` or `--memory` on the
CLI.

Sections:
1. Setup + a demo agent with a realistic 3-day memory
2. The preset catalog (`MEMORY_PRESETS`) and how presets resolve
3. `assemble_context` diffs across presets (`default` / `no_anchor` / `short_memory` / `anchor_ttl2`)
4. The shared `verbatim_window_days` knob and its three window helpers
5. Per-stage overrides

Because every default reproduces the historical hard-wired behaviour
bit-for-bit, the `default` output below is exactly what production runs today.

## 1. Setup + a demo agent

The agent is *detached* (a `MagicMock` environment) and its persona is
monkey-patched, exactly as the unit tests do — this avoids needing the full
YouGov survey data / nation build just to inspect the prompt pipeline.

In [10]:
import os
import sys
from copy import deepcopy
from unittest import mock

# Import the package straight from src/ (no install required).
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

from cag.abm.agent import SurveyedCitizen
from cag.abm.attributes.opinion import ClimatePolicyID, PACKAGE_SCOPE
from cag.abm.config.memory import (
    DEFAULT_MEMORY_CONFIG,
    MEMORY_PRESETS,
    resolve_memory_config,
    resolve_stage_memory,
    verbatim_days,
    summary_upper_exclusive,
    compression_target_day,
)

CARBON_TAX = ClimatePolicyID.CARBON_TAX
RENEWABLES = ClimatePolicyID.RENEWABLE_ENERGY
print("Imported cag.abm.config.memory OK")
print("Available presets:", list(MEMORY_PRESETS))

Imported cag.abm.config.memory OK
Available presets: ['default', 'short_memory', 'wide_memory', 'no_compression', 'no_anchor', 'anchor_ttl2', 'no_own_reasoning', 'reflections_only', 'persona_only']


In [2]:
def build_demo_agent():
    """A detached citizen with a hand-populated 3-day memory across two policies."""
    env = mock.MagicMock()
    c = SurveyedCitizen(agent_id=1, environment=env, year_of_birth=1990)
    # Monkey-patch the persona so we don't need the demographics maps.
    c.get_persona = mock.MagicMock(
        return_value=(
            "I am a 36-year-old secondary-school teacher in Leeds. "
            "I value fairness and worry about the cost of living."
        )
    )

    # Day-0 rationale lives in survey_reasoning[(policy)][0] -> the anchor section.
    c.survey_reasoning[CARBON_TAX] = [
        (0, "A carbon fee is fair only if the dividend really reaches ordinary households."),
        (1, "The broadcast stressed the per-household rebate; I am a little more open."),
        (2, "A neighbour pushed back on admin costs, but I still lean supportive."),
        (3, "Today I feel much the same — cautious support."),
    ]
    c.survey_reasoning[RENEWABLES] = [
        (0, "Renewables are obviously good but I worry about grid reliability."),
        (1, "Someone cited cheap solar; reliability worry is easing."),
    ]

    # Reflections following received messages (days 1..3).
    for day, text in [
        (1, "The politician's rebate framing was persuasive."),
        (2, "A peer's admin-cost point made me hesitate."),
        (3, "On balance the fairness case still holds for me."),
    ]:
        c.reflections.append({
            "day": day, "phase": "P-A", "policy_id": CARBON_TAX,
            "text": text, "messages_received": ["msg"],
        })

    # Compressed gist memory for older days (day 1 becomes a summary at day 3, W=2).
    c.daily_summaries[(1, CARBON_TAX)] = (
        "Day 1: warmed slightly to the carbon fee after the rebate framing."
    )
    return c


agent = build_demo_agent()
print("Demo agent built. Default memory_cfg sections:")
for k, v in agent.memory_cfg.items():
    print(f"  {k}: {v}")

Demo agent built. Default memory_cfg sections:
  persona: {'enabled': True}
  day0_anchor: {'enabled': True, 'ttl_days': None}
  daily_summaries: {'enabled': True}
  recent_reflections: {'enabled': True}
  own_reasoning: {'enabled': True}
  today_so_far: {'enabled': True}
  opinion_trajectory: {'enabled': False}
  verbatim_window_days: 2
  stages: {'peer_message': {}, 'reflection': {}, 'survey': {}}


## 2. The preset catalog

`resolve_memory_config(spec)` accepts `None` (→ default), a **preset name**, or
a literal **dict** of overrides (deep-merged onto the default). It always
returns a fresh, validated config. Below we resolve a few presets and show only
the keys that differ from `default`.

In [3]:
def diff_from_default(cfg):
    """Return the (path -> value) entries where cfg differs from DEFAULT_MEMORY_CONFIG."""
    base = DEFAULT_MEMORY_CONFIG
    out = {}
    def walk(a, b, prefix=""):
        for key in a:
            path = f"{prefix}{key}"
            if isinstance(a[key], dict) and isinstance(b.get(key), dict):
                walk(a[key], b[key], path + ".")
            elif a[key] != b.get(key):
                out[path] = (b.get(key), a[key])
    walk(cfg, base)
    return out


for name in ["default", "short_memory", "wide_memory", "no_compression",
             "no_anchor", "anchor_ttl2", "no_own_reasoning",
             "reflections_only", "persona_only"]:
    cfg = resolve_memory_config(name)
    d = diff_from_default(cfg)
    pretty = ", ".join(f"{p}: {old} -> {new}" for p, (old, new) in d.items()) or "(identical to default)"
    print(f"{name:18s} {pretty}")

default            (identical to default)
short_memory       verbatim_window_days: 2 -> 1
wide_memory        verbatim_window_days: 2 -> 4
no_compression     verbatim_window_days: 2 -> None
no_anchor          day0_anchor.enabled: True -> False
anchor_ttl2        day0_anchor.ttl_days: None -> 2
no_own_reasoning   own_reasoning.enabled: True -> False
reflections_only   day0_anchor.enabled: True -> False, own_reasoning.enabled: True -> False, today_so_far.enabled: True -> False
persona_only       day0_anchor.enabled: True -> False, daily_summaries.enabled: True -> False, recent_reflections.enabled: True -> False, own_reasoning.enabled: True -> False, today_so_far.enabled: True -> False


## 3. `assemble_context` diffs across presets

We ask the agent to assemble the **survey-stage** context for the Carbon Tax on
**day 3**. Attaching a config is exactly what the simulation driver does in
`run()` — it deep-copies the resolved config onto `agent.memory_cfg`.

With the `default` config (2-day verbatim window) you should see, in order:
persona → Day-0 anchor → summary of day 1 → recent reflections (days 2, 3) →
considered position (days 2, 3).

In [4]:
def context_under(preset, day=3, policy=CARBON_TAX, target=None, stage="survey"):
    """Attach a preset and return the assembled context string."""
    a = build_demo_agent()
    a.memory_cfg = deepcopy(resolve_memory_config(preset))
    return a.assemble_context(
        day=day, policy_id=policy,
        target_policy_id=target if target is not None else policy,
        stage=stage,
    )


def show(title, text):
    bar = "=" * 78
    print(f"{bar}\n{title}\n{bar}\n{text}\n")


show("default (2-day window, all sections on)", context_under("default"))

default (2-day window, all sections on)
I am a 36-year-old secondary-school teacher in Leeds. I value fairness and worry about the cost of living.

Original prior position on "Carbon fee and dividend":
A carbon fee is fair only if the dividend really reaches ordinary households.

Summary of recent days:
Day 1: Day 1: warmed slightly to the carbon fee after the rebate framing.

Recent reflections following received messages:
- Day 2: A peer's admin-cost point made me hesitate.
- Day 3: On balance the fairness case still holds for me.

Your considered position in recent days:
- Day 2 — Carbon fee and dividend: A neighbour pushed back on admin costs, but I still lean supportive.
- Day 3 — Carbon fee and dividend: Today I feel much the same — cautious support.



In [5]:
# no_anchor: the 'Original prior position on ...' block disappears.
# Use this to test whether the persistent Day-0 identity tether is what causes
# agents to lock in after Day 1.
show("no_anchor (Day-0 anchor removed)", context_under("no_anchor"))

no_anchor (Day-0 anchor removed)
I am a 36-year-old secondary-school teacher in Leeds. I value fairness and worry about the cost of living.

Summary of recent days:
Day 1: Day 1: warmed slightly to the carbon fee after the rebate framing.

Recent reflections following received messages:
- Day 2: A peer's admin-cost point made me hesitate.
- Day 3: On balance the fairness case still holds for me.

Your considered position in recent days:
- Day 2 — Carbon fee and dividend: A neighbour pushed back on admin costs, but I still lean supportive.
- Day 3 — Carbon fee and dividend: Today I feel much the same — cautious support.



In [6]:
# short_memory: verbatim window shrinks to 1 day. Day 2 moves OUT of the
# verbatim reflections/own-reasoning and (if a summary existed) into the gist.
show("short_memory (1-day verbatim window)", context_under("short_memory"))

short_memory (1-day verbatim window)
I am a 36-year-old secondary-school teacher in Leeds. I value fairness and worry about the cost of living.

Original prior position on "Carbon fee and dividend":
A carbon fee is fair only if the dividend really reaches ordinary households.

Summary of recent days:
Day 1: Day 1: warmed slightly to the carbon fee after the rebate framing.

Recent reflections following received messages:
- Day 3: On balance the fairness case still holds for me.

Your considered position in recent days:
- Day 3 — Carbon fee and dividend: Today I feel much the same — cautious support.



In [7]:
# anchor_ttl2: anchor is present while day <= 2 and retires once day > 2.
show("anchor_ttl2 on day 2 (anchor still present)",
     context_under("anchor_ttl2", day=2))
show("anchor_ttl2 on day 3 (anchor retired)",
     context_under("anchor_ttl2", day=3))

anchor_ttl2 on day 2 (anchor still present)
I am a 36-year-old secondary-school teacher in Leeds. I value fairness and worry about the cost of living.

Original prior position on "Carbon fee and dividend":
A carbon fee is fair only if the dividend really reaches ordinary households.

Recent reflections following received messages:
- Day 1: The politician's rebate framing was persuasive.
- Day 2: A peer's admin-cost point made me hesitate.

Your considered position in recent days:
- Day 1 — Carbon fee and dividend: The broadcast stressed the per-household rebate; I am a little more open.
- Day 2 — Carbon fee and dividend: A neighbour pushed back on admin costs, but I still lean supportive.

anchor_ttl2 on day 3 (anchor retired)
I am a 36-year-old secondary-school teacher in Leeds. I value fairness and worry about the cost of living.

Summary of recent days:
Day 1: Day 1: warmed slightly to the carbon fee after the rebate framing.

Recent reflections following received messages:
- Day 2:

## 4. The shared `verbatim_window_days` knob

One knob (`W`) drives four call sites that must agree: which days are shown
verbatim (§4 reflections, §5 own reasoning), which older days are compressed
into summaries (§3), and which day `manage_memory` compresses. All four derive
from three helpers — the single source of truth. Note `verbatim_days` **includes
day 0**, and `W=None` means *unbounded verbatim / never compress*.

In [8]:
day = 4
print(f"For day={day}:\n")
print(f"{'W':>5} | {'verbatim_days':<22} | {'summaries show d <':<18} | compress day")
print("-" * 70)
for W in [1, 2, 4, None]:
    vd = sorted(verbatim_days(W, day))
    up = summary_upper_exclusive(W, day)
    ct = compression_target_day(W, day)
    print(f"{str(W):>5} | {str(vd):<22} | {str(up):<18} | {ct}")

# Invariant: verbatim and summarised day-sets partition days 1..day with no gap/overlap.
for W in [1, 2, 4]:
    vd = verbatim_days(W, day)
    summ = set(range(1, summary_upper_exclusive(W, day)))
    assert (vd & summ) == set(), f"overlap for W={W}"
    assert (vd | summ) >= set(range(1, day + 1)), f"gap for W={W}"
print("\nPartition invariant holds for W in {1, 2, 4}.")

For day=4:

    W | verbatim_days          | summaries show d < | compress day
----------------------------------------------------------------------
    1 | [4]                    | 4                  | 3
    2 | [3, 4]                 | 3                  | 2
    4 | [1, 2, 3, 4]           | 1                  | None
 None | [0, 1, 2, 3, 4]        | 1                  | None

Partition invariant holds for W in {1, 2, 4}.


## 5. Per-stage overrides

A dict config may carry a `stages` block so a section differs by prompt stage
(`peer_message` / `reflection` / `survey`). `resolve_stage_memory` merges the
stage block over the globals. Here we drop the anchor **only at survey time**
while keeping it during peer messaging.

In [9]:
spec = {"stages": {"survey": {"day0_anchor": {"enabled": False}}}}
cfg = resolve_memory_config(spec)

survey_cfg = resolve_stage_memory(cfg, "survey")
peer_cfg = resolve_stage_memory(cfg, "peer_message")
print("anchor enabled at survey stage:", survey_cfg["day0_anchor"]["enabled"])
print("anchor enabled at peer_message stage:", peer_cfg["day0_anchor"]["enabled"])

a = build_demo_agent()
a.memory_cfg = deepcopy(cfg)
has_anchor_survey = "Original prior position" in a.assemble_context(
    day=3, policy_id=CARBON_TAX, target_policy_id=CARBON_TAX, stage="survey")
has_anchor_peer = "Original prior position" in a.assemble_context(
    day=3, policy_id=CARBON_TAX, target_policy_id=CARBON_TAX, stage="peer_message")
print("anchor block appears in survey context:", has_anchor_survey)
print("anchor block appears in peer context:  ", has_anchor_peer)

anchor enabled at survey stage: False
anchor enabled at peer_message stage: True
anchor block appears in survey context: False
anchor block appears in peer context:   True


## Takeaways

- The whole context pipeline is driven by one `SIM_CONFIG["memory"]` key
  (preset name or dict), or `--memory` on the CLI — no edits to `agent.py`.
- `default` reproduces the pre-v0.8 behaviour bit-for-bit; every other preset
  is an ablation you can drop into an experiment.
- `verbatim_window_days` is the single coupled knob for the verbatim / gist
  memory boundary; `day0_anchor.ttl_days` retires the anchor; per-stage blocks
  let a section differ across peer-message / reflection / survey prompts.
- See `docs/Model_Design.md` §30 and the v0.8 section of `USER_GUIDE.md`.